# Baseline: predict unload duration

This notebook gets you to an honest baseline and one model on a time-ordered split. Cells marked TODO are yours. Run from the kit root.

In [ ]:
import pandas as pd, numpy as np
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error
h = pd.read_csv('data/dock_visit_history.csv', parse_dates=['planned_departure_ts','planned_arrival_ts','slot_start_ts','actual_gate_in_ts'])
print(h.shape)

## 1. Clean

Decide and record each step. The report template asks for the counts.

In [ ]:
before = len(h)
h = h.drop_duplicates()
print('duplicates removed:', before - len(h))
h = h[h.visit_outcome != 'NO_SHOW'].copy()
print('rows with an unload time:', len(h))
# TODO: decide what to do with unload_min > 240 (25 rows). Cap? Remove? Keep and use a robust loss? Write the decision in eval_report_template.md.
# TODO: impute pallet_count where missing. The weight column is a reasonable guide.

## 2. Features that are known before the truck arrives

The arrival hour is known for an unload prediction made at the gate. The yard wait is not. Do not use columns that happen after the moment of prediction.

In [ ]:
h['arrival_hour'] = h.actual_gate_in_ts.dt.hour
h['dow'] = h.actual_gate_in_ts.dt.dayofweek
features = ['pallet_count','load_weight_kg','dock_type','temperature_control_required','arrival_hour','dow','labour_shortage_flag','facility_id','carrier_id']
X = pd.get_dummies(h[features], columns=['dock_type','facility_id','carrier_id'], drop_first=True)
y = h['unload_min']
order = h.actual_gate_in_ts.argsort()
X, y, h = X.iloc[order], y.iloc[order], h.iloc[order]

## 3. Baseline: median unload per dock type

If your model cannot beat this, the model is not the story.

In [ ]:
cut = h.actual_gate_in_ts >= '2026-07-01'
train, test = ~cut, cut
med = h[train].groupby('dock_type')['unload_min'].median()
base_pred = h.loc[test,'dock_type'].map(med)
print('baseline MAE (minutes):', round(mean_absolute_error(y[test], base_pred), 1))

## 4. One model, same split

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor
m = HistGradientBoostingRegressor(max_iter=300, learning_rate=0.05, random_state=0)
m.fit(X[train], y[train])
pred = m.predict(X[test])
print('model MAE (minutes):', round(mean_absolute_error(y[test], pred), 1))
# TODO: add a quantile model (loss='quantile', quantile=0.9) to get the p90 the integration contract needs.

## 5. Cross-validation that respects time

In [ ]:
tss = TimeSeriesSplit(n_splits=4)
maes = []
for tr, te in tss.split(X):
    m.fit(X.iloc[tr], y.iloc[tr]); maes.append(mean_absolute_error(y.iloc[te], m.predict(X.iloc[te])))
print('fold MAEs:', [round(v,1) for v in maes])

## 6. What drives it

TODO: permutation importance, then translate the top five into business sentences in the report. Example: 'every extra pallet adds about a minute and a half'.

In [ ]:
from sklearn.inspection import permutation_importance
m.fit(X[train], y[train])
pi = permutation_importance(m, X[test], y[test], n_repeats=5, random_state=0)
pd.Series(pi.importances_mean, index=X.columns).sort_values(ascending=False).head(10).round(2)

## 7. Where it is wrong

TODO: list the five largest errors on the test set with their rows. Explain at least two.

In [ ]:
err = (y[test] - pred).abs().sort_values(ascending=False).head(5)
h.loc[err.index, ['visit_id','dock_type','pallet_count','arrival_hour','labour_shortage_flag','unload_min']].assign(pred=pred[[list(y[test].index).index(i) for i in err.index]].round(0))